# NeuroSymbolic-T4: Full Research Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/NeuroSymbolic-T4/blob/main/notebooks/NeuroSymbolic_T4_Full_Pipeline.ipynb)

**Complete end-to-end research pipeline: Download → Train → Benchmark → Visualize.**

This notebook provides a rigorous, automated workflow for reproducing the NeuroSymbolic-T4 results on Google Colab T4 GPUs.

## 📋 Pipeline Overview
1. **Environment Setup**: Install dependencies and verify GPU
2. **Data Acquisition**: Automatically download and prepare CLEVR Mini
3. **Model Training**: Train the neurosymbolic system with mixed precision
4. **Rigorous Benchmarking**: Run the full ICML evaluation suite on trained model
5. **Result Analysis**: Generate LaTeX tables and publication figures
6. **Artifact Export**: Save model checkpoints and research results

## 1. Environment Setup

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone and install
!git clone https://github.com/Tommaso-R-Marena/NeuroSymbolic-T4.git
%cd NeuroSymbolic-T4
!pip install -q -r requirements.txt
!pip install -q wandb

import sys
sys.path.append('.')
print("\n✅ Environment ready!")

## 2. Data Acquisition

In [ ]:
import shutil
stat = shutil.disk_usage('.')
print(f"Available disk space: {stat.free/1e9:.1f}GB")

# Download CLEVR mini (~1.5GB)
!python benchmarks/download_datasets.py --dataset clevr_mini --data-root ./data

print("\n✅ Dataset ready at ./data/CLEVR_mini")

## 3. Model Training

Training the model for 20 epochs with mixed precision (AMP) and EfficientNet backbone.

In [ ]:
!python train_benchmarks.py \
    --dataset clevr \
    --clevr-root ./data/CLEVR_mini \
    --epochs 20 \
    --batch-size 32 \
    --use-amp \
    --output-dir ./checkpoints

## 4. Rigorous Benchmarking

Running the full evaluation suite on the best saved checkpoint.

In [ ]:
!python benchmarks/run_all.py \
    --checkpoint ./checkpoints/best_model.pt \
    --clevr-root ./data/CLEVR_mini \
    --output-dir ./results

## 5. Result Analysis & Visualization

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")

# Load results
results_path = Path('./results/benchmark_results.json')
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    
    print("\nCLEVR Benchmark Results:")
    clevr = results.get('CLEVR', {})
    print(f"  Accuracy: {clevr.get('accuracy', 0):.2%}")
    print(f"  Reasoning Depth: {clevr.get('avg_reasoning_depth', 0):.2f}")

    # Generate Figures
    !python paper/prepare_figures.py --results-dir ./results --output-dir ./figures
    
    # Show LaTeX table
    print("\nGenerated LaTeX Table:")
    with open('./results/results_table.tex') as f:
        print(f.read())
else:
    print("⚠️ Results not found. Check if benchmarking step succeeded.")

## 6. Artifact Export

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
    export_path = '/content/drive/MyDrive/NeuroSymbolic_T4_Results'
    !mkdir -p {export_path}
    !cp -r ./checkpoints ./results ./figures {export_path}
    print(f"\n✅ Artifacts exported to {export_path}")
except:
    print("\n⚠️ Google Drive not mounted. Results are saved locally in the Colab runtime.")